In [20]:
import pandas as pd
import numpy as np

In [21]:
# --- Parameters ---
N = 100  # number of cases
SOC_high = 80.0  # nominal high SOC target (%)
SOC_low = 20.0   # nominal low SOC target (%)
SOH_start = 100.0
SOH_end = 90.0

# --- Initialize arrays ---
steps = np.arange(1, N + 1)
cases = np.array(["Discharge" if i % 2 else "Charge" for i in range(1, N + 1)])
start_temp = np.full(N, 25.0)
start_soc = np.zeros(N)
end_soc = np.zeros(N)

# --- Temperature variation pattern ---
# First 10 steps = 25°C, rest random from [0, 10, 25, 35, 45]
temp_choices = [0.0, 10.0, 25.0, 35.0, 45.0]
start_temp[:10] = 25.0
start_temp[10:] = np.random.choice(temp_choices, size=N - 10, replace=True)

In [22]:
# --- SOH linear decrease ---
soh = np.round(SOH_start + (SOH_end - SOH_start) * (steps - 1) / (N - 1), 1)

# --- SOC imbalance: first 10 cycles = 0, rest random 0–5 ---
soc_imbalance = np.zeros(N)
soc_imbalance[:10] = 0.0
soc_imbalance[10:] = np.round(np.random.uniform(0.0, 5.0, size=N - 10))

# --- Define SOC transitions ---
for k in range(N):
    if k == 0:
        # First step (Discharge): full cycle 100 → 0
        start_soc[k] = 100.0
        end_soc[k] = 0.0
    elif k == 1:
        # Second step (Charge): full cycle 0 → 100
        start_soc[k] = 0.0
        end_soc[k] = 100.0
    else:
        # From step 3 onwards:
        start_soc[k] = end_soc[k - 1]
        if cases[k] == "Discharge":
            # Discharge: end between 5 and 20 %
            end_soc[k] = np.round(np.random.uniform(5, 20))
        else:
            # Charge: end between 80 % and 95
            end_soc[k] = np.round(np.random.uniform(80, 95))

# --- Clamp SOC values to 0–100 range ---
end_soc = np.clip(end_soc, 0, 100)
start_soc = np.clip(start_soc, 0, 100)

In [26]:
# --- Create dataframe ---
df = pd.DataFrame({
    "Step": steps,
    "Case": cases,
    "StartTemp_C": start_temp,
    "StartSOC_avg": np.round(start_soc),
    "EndSOC_avg": np.round(end_soc),
    "SOC_imbalance": np.round(soc_imbalance),
    "SOH_avg": np.round(soh, 1)
})
df

,Step,Case,StartTemp_C,StartSOC_avg,EndSOC_avg,SOC_imbalance,SOH_avg
0,1,Discharge,25.0,100.0,0.0,0.0,100.0
1,2,Charge,25.0,0.0,100.0,0.0,99.9
2,3,Discharge,25.0,100.0,12.0,0.0,99.8
3,4,Charge,25.0,12.0,89.0,0.0,99.7
4,5,Discharge,25.0,89.0,11.0,0.0,99.6
...,...,...,...,...,...,...,...
95,96,Charge,35.0,17.0,85.0,1.0,90.4
96,97,Discharge,35.0,85.0,19.0,3.0,90.3
97,98,Charge,10.0,19.0,82.0,4.0,90.2
98,99,Discharge,45.0,82.0,13.0,1.0,90.1


In [27]:

# --- Save to CSV ---
df.to_excel(r"C:\Users\mmackenzie\OneDrive - ZELEROS GLOBAL S.L\Zeleros - Zeleros\Operaciones\4- E-drive\05- Projects\BMS\State Of Art\Synthetic data\simulation_cases.xlsx", index=False, sheet_name="Simulation cases")
print(df.head(15))
print("\n✅ Simulation table with randomized SOC windows saved to 'simulation_cases.xlsx'")

    Step       Case  StartTemp_C  StartSOC_avg  EndSOC_avg  SOC_imbalance  SOH_avg
0      1  Discharge         25.0         100.0         0.0            0.0    100.0
1      2     Charge         25.0           0.0       100.0            0.0     99.9
2      3  Discharge         25.0         100.0        12.0            0.0     99.8
3      4     Charge         25.0          12.0        89.0            0.0     99.7
4      5  Discharge         25.0          89.0        11.0            0.0     99.6
5      6     Charge         25.0          11.0        92.0            0.0     99.5
6      7  Discharge         25.0          92.0        15.0            0.0     99.4
7      8     Charge         25.0          15.0        80.0            0.0     99.3
8      9  Discharge         25.0          80.0        13.0            0.0     99.2
9     10     Charge         25.0          13.0        94.0            0.0     99.1
10    11  Discharge          0.0          94.0        18.0            5.0     99.0
11  